In [1]:
import time
import threading
import ipywidgets.widgets as widgets
from jetank import Robot
from jetank import TTLServo
from jetcam.csi_camera import CSICamera

# ==============================================================================
# 1. INITIALIZE ROBOT & HARDWARE
# ==============================================================================
print("Initializing Jetank hardware...")
robot = Robot()

# ==============================================================================
# 2. INITIALIZE CAMERA
# ==============================================================================
print("Initializing CSI Camera...")
try:
    # 224x224 is standard for Jetank's default AI models
    camera = CSICamera(width=224, height=224, capture_fps=30)
    warmup_image = camera.read()
    print(f"-> Camera active. Image frame shape: {warmup_image.shape}")
except Exception as e:
    print(f"-> Camera initialization failed: {e}")
    print("-> Continuing script without camera feed...")

# ==============================================================================
# 3. INITIALIZE CONTROLLER
# ==============================================================================
print("Connecting to gamepad controller...")
# Index 0 targets the first connected USB/Bluetooth controller
controller = widgets.Controller(index=0)
display(controller)
print("-> Controller initialized. Please interact with it to verify connection.")

# ==============================================================================
# 4. INITIAL SERVO TEST
# ==============================================================================
print("Testing Servo 5 initialization position...")
# Sending your requested baseline positioning command
# Format: (Servo ID, Angle, Mode/Speed Config, Duration)
TTLServo.servoAngleCtrl(5, 0, 1, 100)
time.sleep(1.0)
print("-> Servo 5 set to 0 degrees.")

# ==============================================================================
# 5. DYNAMIC CONTROL LOOP THREAD
# ==============================================================================
def jetank_control_loop():
    print("\n[Control Loop Active]")
    print("Use the Left Joystick (Y-axis) to dynamically adjust Servo 5.")
    print("Press the 'A' button (Button 0) on your controller to stop.")
    
    # Track the last sent value to avoid flooding the serial bus with identical data
    last_angle = None
    
    while True:
        try:
            # controller.axes[1] targets the Left Joystick vertical movement (Y-axis)
            # Returns a float between -1.0 and 1.0
            joystick_y = controller.axes[1].value
            
            # Map the joystick's -1.0 to 1.0 range to a -90 to 90 degree movement range
            # Adjust the multiplier if you need a narrower or wider physical range
            target_angle = int(joystick_y * 90)
            
            # Only send serial commands if the joystick position actually changed
            if target_angle != last_angle:
                TTLServo.servoAngleCtrl(5, target_angle, 1, 50)
                last_angle = target_angle
            
            # Break condition: Safely stop the loop if Button 0 ('A' button) is held
            if controller.buttons[0].value:
                print("\n[Control Loop stopped by user request]")
                # Return servo to default 0 before exiting
                TTLServo.servoAngleCtrl(5, 0, 1, 100)
                break
                
            # 20Hz refresh rate (50ms sleep) keeps the robot highly responsive
            # without choking the serial communication bus
            time.sleep(0.05)
            
        except KeyboardInterrupt:
            print("\n[Control Loop intercepted by notebook kernel]")
            break
        except Exception as loop_error:
            print(f"Error within runtime loop: {loop_error}")
            break

# Execute the control loop in a background thread so the Jupyter Notebook
# interface remains responsive and unblocked
control_thread = threading.Thread(target=jetank_control_loop)
control_thread.daemon = True # Allows thread to close automatically when kernel stops
control_thread.start()

ModuleNotFoundError: No module named 'jetank'